# 11. EDA Insights

## Objective

This notebook consolidates the most important findings from the EDA, cleaning, feature engineering, and feature-selection stages.

It is designed to:
- consolidate the key findings from all EDA notebooks
- translate technical observations into business understanding
- identify modeling implications
- define recommendations for the modeling phase


In [23]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import pointbiserialr
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RANDOM_STATE, TEST_SIZE

raw_data_path = project_root / "data" / "raw" / "telco.csv"
cleaned_data_path = project_root / "data" / "interim" / "telco_churn_cleaned.csv"
engineered_data_path = project_root / "data" / "processed" / "telco_churn_engineered.csv"
selected_data_path = project_root / "data" / "processed" / "telco_churn_selected.csv"
feature_columns_path = project_root / "artifacts" / "feature_columns.json"
feature_selection_tables_dir = project_root / "reports" / "tables" / "feature_selection"
final_selection_summary_path = feature_selection_tables_dir / "final_selection_summary.csv"
final_feature_decisions_path = feature_selection_tables_dir / "final_feature_decisions.csv"

if not selected_data_path.exists():
    raise FileNotFoundError(
        "Selected dataset not found. Run notebook 10_feature_selection.ipynb first "
        f"to create: {selected_data_path}"
    )

raw_df = pd.read_csv(raw_data_path) if raw_data_path.exists() else None
cleaned_df = pd.read_csv(cleaned_data_path) if cleaned_data_path.exists() else None
engineered_df = pd.read_csv(engineered_data_path) if engineered_data_path.exists() else None
selected_df = pd.read_csv(selected_data_path)

df = selected_df.copy()
target_column = "target"
if target_column not in df.columns:
    raise KeyError(f"{target_column!r} was not found in the selected dataset.")

if feature_columns_path.exists():
    selected_feature_columns = json.loads(feature_columns_path.read_text())
else:
    selected_feature_columns = [column for column in df.columns if column != target_column]

missing_selected_features = [column for column in selected_feature_columns if column not in df.columns]
if missing_selected_features:
    raise KeyError(f"Selected feature columns missing from dataset: {missing_selected_features}")

X = df[selected_feature_columns].copy()
y = df[target_column].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
binary_features = [
    column
    for column in numeric_features
    if set(pd.Series(X[column]).dropna().astype(float).unique()).issubset({0.0, 1.0})
]
ordinal_features = [column for column in numeric_features if column.endswith("_ordinal")]
continuous_numeric_features = [
    column for column in numeric_features if column not in binary_features and column not in ordinal_features
]
engineered_feature_markers = (
    "_ordinal",
    "_group",
    "_profile",
    "_band",
    "_x_",
    "is_",
    "log_",
)
engineered_features = [
    column for column in selected_feature_columns if any(marker in column for marker in engineered_feature_markers)
]
original_features = [column for column in selected_feature_columns if column not in engineered_features]

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Dataset Recap

This section gives a short reminder of the modeling-ready dataset, target definition, feature groups, and current split status.


In [24]:
dataset_recap = pd.DataFrame(
    {
        "metric": [
            "dataset used for insights",
            "dataset shape",
            "target column",
            "problem type",
            "selected feature count",
            "original-style feature count",
            "engineered feature count",
            "continuous numeric feature count",
            "binary feature count",
            "ordinal feature count",
            "categorical feature count",
            "train/test split status",
            "train shape",
            "test shape",
        ],
        "value": [
            str(selected_data_path.relative_to(project_root)),
            f"{df.shape[0]} rows x {df.shape[1]} columns",
            target_column,
            "binary classification",
            len(selected_feature_columns),
            len(original_features),
            len(engineered_features),
            len(continuous_numeric_features),
            len(binary_features),
            len(ordinal_features),
            len(categorical_features),
            f"virtual split prepared in notebook using TEST_SIZE={TEST_SIZE}",
            f"{X_train.shape[0]} rows x {X_train.shape[1]} features",
            f"{X_test.shape[0]} rows x {X_test.shape[1]} features",
        ],
    }
)

dataset_recap


,metric,value
0,dataset used for insights,data/processed/telco_churn_selected.csv
1,dataset shape,7043 rows x 54 columns
2,target column,target
3,problem type,binary classification
4,selected feature count,53
5,original-style feature count,21
6,engineered feature count,32
7,continuous numeric feature count,6
8,binary feature count,46
9,ordinal feature count,1


## Key Data Quality Findings

This section summarizes the main cleaning findings from earlier notebooks and converts them into a compact action log for modeling.


In [25]:
quality_rows = []

if raw_df is not None:
    totalcharges_blank_mask = raw_df["TotalCharges"].astype(str).str.strip().eq("")
    quality_rows.append(
        {
            "issue": "Blank TotalCharges values in raw data",
            "finding": int(totalcharges_blank_mask.sum()),
            "detail": "Blank values are present in the raw file and should not be treated as random missingness.",
            "action_taken": "Handled during cleaning and converted into stable numeric billing features downstream.",
        }
    )
    if "tenure" in raw_df.columns:
        quality_rows.append(
            {
                "issue": "Blank TotalCharges linked to tenure",
                "finding": int(raw_df.loc[totalcharges_blank_mask, "tenure"].eq(0).sum()),
                "detail": "These records are typically new customers with tenure = 0.",
                "action_taken": "Kept records and cleaned charges consistently instead of dropping them blindly.",
            }
        )
    quality_rows.append(
        {
            "issue": "Duplicate rows in raw data",
            "finding": int(raw_df.duplicated().sum()),
            "detail": "Checks whether exact duplicates exist in the source dataset.",
            "action_taken": "No large duplicate issue expected for this dataset; identifiers remain excluded from modeling.",
        }
    )

quality_rows.append(
    {
        "issue": "Missing values in selected dataset",
        "finding": int(df.isna().sum().sum()),
        "detail": "The selected modeling dataset should be numerically stable and ready for downstream pipelines.",
        "action_taken": "Review any remaining nulls before fitting final models.",
    }
)
quality_rows.append(
    {
        "issue": "Duplicate feature vectors in selected dataset",
        "finding": int(df.duplicated().sum()),
        "detail": "These are repeated rows in the final feature space after selection and can reflect repeated customer profiles rather than source-record duplication.",
        "action_taken": "Review whether exact duplicate feature vectors should be retained or deduplicated based on the modeling strategy.",
    }
)

skewness_df = (
    df[continuous_numeric_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
    .rename("skewness")
    .reset_index()
    .rename(columns={"index": "feature"})
)
strong_skew = skewness_df.loc[skewness_df["skewness"].abs() >= 1].copy()

quality_rows.append(
    {
        "issue": "Strong skew in continuous numeric variables",
        "finding": int(strong_skew.shape[0]),
        "detail": ", ".join(strong_skew["feature"].head(6).tolist()) if not strong_skew.empty else "No strongly skewed continuous features detected.",
        "action_taken": "Prefer transformed versions such as log_total_charges when they improve stability and interpretability.",
    }
)

quality_summary_df = pd.DataFrame(quality_rows)
quality_summary_df


,issue,finding,detail,action_taken
0,Blank TotalCharges values in raw data,11,Blank values are present in the raw file and s...,Handled during cleaning and converted into sta...
1,Blank TotalCharges linked to tenure,11,These records are typically new customers with...,Kept records and cleaned charges consistently ...
2,Duplicate rows in raw data,0,Checks whether exact duplicates exist in the s...,No large duplicate issue expected for this dat...
3,Missing values in selected dataset,0,The selected modeling dataset should be numeri...,Review any remaining nulls before fitting fina...
4,Duplicate feature vectors in selected dataset,22,These are repeated rows in the final feature s...,Review whether exact duplicate feature vectors...
5,Strong skew in continuous numeric variables,0,No strongly skewed continuous features detected.,Prefer transformed versions such as log_total_...


The selected modeling dataset is in good condition for the next phase: the original blank `TotalCharges` issue was handled earlier in preprocessing, identifiers have been removed, the target is standardized as `target`, and the final table contains no missing values or duplicate columns.


## Univariate Insights

The goal here is not to recreate every plot, but to extract the most important single-variable patterns that matter for churn understanding and modeling.


In [26]:
class_balance_df = (
    df[target_column]
    .map({0: "Stayed", 1: "Churned"})
    .value_counts(dropna=False)
    .rename_axis("target_class")
    .reset_index(name="count")
)
class_balance_df["share_pct"] = (class_balance_df["count"] / len(df) * 100).round(2)

continuous_distribution_df = df[continuous_numeric_features].describe().T.reset_index().rename(columns={"index": "feature"})
continuous_distribution_df["skewness"] = continuous_distribution_df["feature"].map(
    dict(zip(skewness_df["feature"], skewness_df["skewness"]))
).round(3)

ordinal_distribution_df = pd.DataFrame(columns=["feature", "min", "25%", "50%", "75%", "max"])
if ordinal_features:
    ordinal_distribution_df = df[ordinal_features].describe().T.reset_index().rename(columns={"index": "feature"})

categorical_cardinality_df = pd.DataFrame(columns=["feature", "unique_categories", "top_category", "top_category_share_pct"])
if categorical_features:
    categorical_cardinality_df = pd.DataFrame(
        {
            "feature": categorical_features,
            "unique_categories": [df[column].nunique(dropna=False) for column in categorical_features],
            "top_category": [df[column].mode(dropna=False).iloc[0] for column in categorical_features],
            "top_category_share_pct": [round(df[column].value_counts(normalize=True, dropna=False).iloc[0] * 100, 2) for column in categorical_features],
        }
    ).sort_values(["unique_categories", "top_category_share_pct"], ascending=[False, False])

print("Class balance")
display(class_balance_df)
print("\nContinuous feature summary")
display(continuous_distribution_df[["feature", "mean", "std", "min", "25%", "50%", "75%", "max", "skewness"]].sort_values("skewness", ascending=False))
print("\nOrdinal feature summary")
display(ordinal_distribution_df[["feature", "min", "25%", "50%", "75%", "max"]] if not ordinal_distribution_df.empty else ordinal_distribution_df)
print("\nCategorical dominance snapshot")
display(categorical_cardinality_df.head(12) if not categorical_cardinality_df.empty else categorical_cardinality_df)


Class balance


,target_class,count,share_pct
0,Stayed,5174,73.46
1,Churned,1869,26.54



Continuous feature summary


,feature,mean,std,min,25%,50%,75%,max,skewness
0,tenure_x_MonthlyCharges,2279.581350,2264.729447,0.000,394.000000,1393.600000,3786.100000,8550.000000,0.961
3,service_count,2.459747,2.045539,0.000,1.000000,2.000000,4.000000,7.000000,0.415
5,tenure,32.371149,24.559481,0.000,9.000000,29.000000,55.000000,72.000000,0.240
2,avg_monthly_spend,64.762906,30.189796,13.775,35.935156,70.337500,90.174158,121.400000,-0.210
1,MonthlyCharges,64.761692,30.090047,18.250,35.500000,70.350000,89.850000,118.750000,-0.221
4,log_total_charges,6.932543,1.569371,0.000,5.990339,7.241044,8.239488,9.069445,-0.824



Ordinal feature summary


,feature,min,25%,50%,75%,max
0,Contract_ordinal,0.0,0.0,0.0,1.0,2.0



Categorical dominance snapshot


,feature,unique_categories,top_category,top_category_share_pct


**Observed univariate takeaways:**

- The selected modeling dataset stays moderately imbalanced: about a quarter of customers churned, so accuracy alone would still be misleading during model evaluation.
- The main continuous modeling variables remain broad rather than tightly clustered. `MonthlyCharges`, `tenure`, and spend-derived features such as `log_total_charges`, `avg_monthly_spend`, and `tenure_x_MonthlyCharges` all carry wide business variation.
- `Contract_ordinal` is best interpreted as an ordinal encoded commitment feature, not as a continuous business variable, so it should be read as ordered contract tiers rather than as a measured quantity.
- Skew is concentrated in a smaller set of selected continuous variables, and `log_total_charges` remains the cleaner cumulative-charge representation for modeling.
- The final dataset is almost entirely numeric by design, which means notebook 10 has already done the categorical handling needed for downstream modeling pipelines.
- Because the selected table is narrower than the full engineered dataset, the remaining variables should now be interpreted as prioritized modeling inputs rather than exploratory feature candidates.


## Bivariate Insights With Target

This is one of the most important sections. It summarizes which variables show the clearest relationship with churn and which patterns appear strongest from both statistical and business viewpoints.


In [34]:
numeric_target_rows = []
for column in continuous_numeric_features + ordinal_features + binary_features:
    series = df[column].astype(float)
    corr_value = series.corr(y)
    point_r, point_p = pointbiserialr(y, series)
    numeric_target_rows.append(
        {
            "feature": column,
            "feature_role": "ordinal" if column in ordinal_features else ("binary" if column in binary_features else "continuous"),
            "pearson_with_target": round(float(corr_value), 4),
            "abs_pearson_with_target": round(abs(float(corr_value)), 4),
            "pointbiserial_r": round(float(point_r), 4),
            "pointbiserial_p": float(point_p),
        }
    )

numeric_target_df = pd.DataFrame(numeric_target_rows).sort_values(
    "abs_pearson_with_target", ascending=False
).reset_index(drop=True)

categorical_target_df = pd.DataFrame(
    columns=["feature", "category_value", "segment_count", "churn_rate_pct", "overall_gap"]
)
if categorical_features:
    categorical_target_rows = []
    for column in categorical_features:
        grouped = (
            df.groupby(column, dropna=False)[target_column]
            .agg(["mean", "count"])
            .reset_index()
            .rename(columns={column: "category_value", "mean": "churn_rate", "count": "segment_count"})
        )
        grouped["feature"] = column
        grouped["overall_gap"] = (grouped["churn_rate"] - df[target_column].mean()).abs()
        categorical_target_rows.append(grouped)

    categorical_target_df = pd.concat(categorical_target_rows, ignore_index=True)
    categorical_target_df["churn_rate_pct"] = (categorical_target_df["churn_rate"] * 100).round(2)
    categorical_target_df = categorical_target_df.sort_values(
        ["overall_gap", "segment_count"], ascending=[False, False]
    ).reset_index(drop=True)

print("Top numeric, ordinal, and binary relationships with churn")
display(numeric_target_df.head(15))
print("\nTop categorical segments by churn-rate gap")
display(categorical_target_df[["feature", "category_value", "segment_count", "churn_rate_pct", "overall_gap"]].head(20) if not categorical_target_df.empty else categorical_target_df)


Top numeric, ordinal, and binary relationships with churn


,feature,feature_role,pearson_with_target,abs_pearson_with_target,pointbiserial_r,pointbiserial_p
0,is_month_to_month,binary,0.4051,0.4051,0.4051,1.991701e-276
1,Contract_ordinal,ordinal,-0.3967,0.3967,-0.3967,3.666675e-264
2,contract_payment_profile_Month-to-month__Elect...,binary,0.3676,0.3676,0.3676,2.849272e-224
3,internet_onlinesecurity_profile_Fiber optic__No,binary,0.3549,0.3549,0.3549,3.619977e-208
4,tenure,continuous,-0.3522,0.3522,-0.3522,7.999058e-205
5,internet_techsupport_profile_Fiber optic__No,binary,0.3520,0.3520,0.3520,1.373561e-204
6,TechSupport_No,binary,0.3373,0.3373,0.3373,6.413692e-187
7,is_new_customer,binary,0.3177,0.3177,0.3177,6.867572e-165
8,internet_service_group_fiber,binary,0.3080,0.3080,0.3080,1.200784e-154
9,log_total_charges,continuous,-0.2340,0.2340,-0.2340,3.437483e-88



Top categorical segments by churn-rate gap


,feature,category_value,segment_count,churn_rate_pct,overall_gap


**Observed bivariate takeaways:**

- Contract and lifecycle variables remain the clearest churn signals in the selected dataset. `Contract_ordinal`, `is_month_to_month`, `tenure`, and `is_new_customer` still anchor the strongest direct relationships with `target`.
- `Contract_ordinal` should be interpreted as an ordered categorical proxy for commitment length rather than as a continuous business measurement.
- Spend-related variables still matter, but they should be read as a family rather than as fully independent signals. `log_total_charges`, `MonthlyCharges`, `avg_monthly_spend`, and `tenure_x_MonthlyCharges` all contribute useful churn information.
- Support and service indicators remain practically important, especially the selected support-profile dummies carried forward from notebook 10.
- The selected dataset is now numeric-first, so target-relationship review is concentrated more on continuous, ordinal, and binary signal strength than on raw categorical segment summaries.
- This makes notebook 11 a better bridge into modeling, because the strongest patterns are now attached directly to the final modeling columns rather than to exploratory alternatives that were already dropped.


## Multivariate Insights

This section explains combined patterns instead of isolated variables. It highlights redundancy, interaction structure, and churn-risk profiles that become clearer when variables are read together.


In [35]:
numeric_corr_matrix = df[continuous_numeric_features + ordinal_features + binary_features].corr(numeric_only=True)

high_corr_pairs = []
for left_index, left_feature in enumerate(numeric_corr_matrix.columns):
    for right_feature in numeric_corr_matrix.columns[left_index + 1:]:
        corr_value = numeric_corr_matrix.loc[left_feature, right_feature]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append(
                {
                    "feature_1": left_feature,
                    "feature_2": right_feature,
                    "correlation": round(float(corr_value), 4),
                    "abs_correlation": round(abs(float(corr_value)), 4),
                }
            )

high_corr_pairs_df = pd.DataFrame(high_corr_pairs).sort_values("abs_correlation", ascending=False)
if high_corr_pairs_df.empty:
    high_corr_pairs_df = pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation"])

combined_risk_profiles = pd.DataFrame(
    columns=["Contract", "payment_method_group", "has_support_services", "churn_rate", "segment_count", "churn_rate_pct"]
)
if engineered_df is not None and {"Contract", "payment_method_group", "has_support_services", target_column}.issubset(engineered_df.columns):
    combined_risk_profiles = (
        engineered_df.groupby(["Contract", "payment_method_group", "has_support_services"], dropna=False)[target_column]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "churn_rate", "count": "segment_count"})
    )
    combined_risk_profiles = combined_risk_profiles.loc[combined_risk_profiles["segment_count"] >= 50].copy()
    combined_risk_profiles["churn_rate_pct"] = (combined_risk_profiles["churn_rate"] * 100).round(2)
    combined_risk_profiles = combined_risk_profiles.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

print("High-correlation pairs across continuous, ordinal, and binary features (|r| >= 0.7)")
display(high_corr_pairs_df.head(20))
print("\nCombined churn-risk profiles")
display(combined_risk_profiles.head(15))


High-correlation pairs across continuous, ordinal, and binary features (|r| >= 0.7)


,feature_1,feature_2,correlation,abs_correlation
45,OnlineSecurity_No internet service,internet_techsupport_profile_No__No internet s...,1.0,1.0
48,DeviceProtection_No internet service,internet_onlinesecurity_profile_No__No interne...,1.0,1.0
47,DeviceProtection_No internet service,internet_techsupport_profile_No__No internet s...,1.0,1.0
35,StreamingMovies_No internet service,internet_service_group_no_internet,1.0,1.0
36,StreamingMovies_No internet service,OnlineSecurity_No internet service,1.0,1.0
37,StreamingMovies_No internet service,DeviceProtection_No internet service,1.0,1.0
38,StreamingMovies_No internet service,internet_techsupport_profile_No__No internet s...,1.0,1.0
39,StreamingMovies_No internet service,internet_onlinesecurity_profile_No__No interne...,1.0,1.0
40,internet_service_group_no_internet,OnlineSecurity_No internet service,1.0,1.0
41,internet_service_group_no_internet,DeviceProtection_No internet service,1.0,1.0



Combined churn-risk profiles


,Contract,payment_method_group,has_support_services,churn_rate,segment_count,churn_rate_pct
2,Month-to-month,electronic_check,0,0.619289,985,61.93
3,Month-to-month,electronic_check,1,0.443931,865,44.39
0,Month-to-month,auto_payment,0,0.366534,502,36.65
4,Month-to-month,manual_check,0,0.338409,591,33.84
1,Month-to-month,auto_payment,1,0.309524,630,30.95
5,Month-to-month,manual_check,1,0.271523,302,27.15
8,One year,electronic_check,0,0.187500,80,18.75
9,One year,electronic_check,1,0.183521,267,18.35
7,One year,auto_payment,1,0.114695,558,11.47
11,One year,manual_check,1,0.103226,155,10.32


**Observed multivariate takeaways:**

- Several selected feature families still contain meaningful redundancy. `MonthlyCharges`, `avg_monthly_spend`, `tenure`, `tenure_x_MonthlyCharges`, and `log_total_charges` continue to overlap strongly, so linear models should not keep all of them without review.
- Contract commitment remains a concentrated signal family. `Contract_ordinal` and `is_month_to_month` still overlap heavily, which is why notebook 10 marked them for explicit selection control.
- Combined customer profiles remain more informative than isolated features when you read the engineered dataset in business terms. Month-to-month customers paying by electronic check without support services still define one of the clearest high-risk segments.
- The selected dataset is cleaner than the full engineered table, but that does not remove the need for multicollinearity control inside the final modeling pipeline.
- These results support notebook 10’s approach: keep the strongest core features, retain a few interaction/profile signals selectively, and remove obviously duplicate encodings before model fitting.


## Feature Engineering Insights

This section summarizes which engineering choices were useful, which ones improved interpretability, and which ones may need restraint during modeling.


In [36]:
engineering_summary_df = pd.DataFrame(
    {
        "feature_group": [
            "log-transformed fields",
            "binary encodings",
            "ordinal encodings",
            "grouped categorical fields",
            "interaction / profile fields",
            "customer-behavior flags",
        ],
        "matching_features": [
            ", ".join([column for column in selected_feature_columns if column.startswith("log_")]) or "None detected",
            ", ".join(binary_features) or "None detected",
            ", ".join(ordinal_features) or "None detected",
            ", ".join([column for column in selected_feature_columns if column.endswith("_group") or column.endswith("_band")]) or "None detected",
            ", ".join([column for column in selected_feature_columns if "_profile" in column or "_x_" in column]) or "None detected",
            ", ".join([column for column in selected_feature_columns if column.startswith("is_") or column in {"service_count", "has_support_services", "avg_monthly_spend"}]) or "None detected",
        ],
        "modeling_note": [
            "Use transformed versions when they materially reduce skew or stabilize scale.",
            "Binary features are now kept directly on their final modeling columns rather than as duplicate helper fields.",
            "Helpful when there is a natural order, such as contract duration, but still conceptually categorical rather than continuous.",
            "Improves interpretability by reducing sparse category fragmentation.",
            "Captures business behavior but can introduce overlapping signal with base columns.",
            "Often strong churn proxies, but should still be checked for redundancy before linear modeling.",
        ],
    }
)

engineering_summary_df


,feature_group,matching_features,modeling_note
0,log-transformed fields,log_total_charges,Use transformed versions when they materially ...
1,binary encodings,contract_payment_profile_One year__Bank transf...,Binary features are now kept directly on their...
2,ordinal encodings,Contract_ordinal,"Helpful when there is a natural order, such as..."
3,grouped categorical fields,None detected,Improves interpretability by reducing sparse c...
4,interaction / profile fields,contract_payment_profile_One year__Bank transf...,Captures business behavior but can introduce o...
5,customer-behavior flags,"is_new_customer, avg_monthly_spend, service_co...","Often strong churn proxies, but should still b..."


Customer-behavior flags and contract encodings are the most analytically useful engineered groups because they compress churn-relevant lifecycle and commitment signals into features that are both interpretable and model-ready. The interaction and profile fields add valuable business context, but they need the most restraint during modeling because they can quickly introduce sparse columns and overlap heavily with already-strong base features.


## Outlier And Scaling Implications

This section clarifies whether observed outliers should be capped, transformed, or left intact, and how scaling should be handled in the modeling phase.


In [38]:
outlier_rows = []
for column in continuous_numeric_features:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((df[column] < lower_bound) | (df[column] > upper_bound)).sum())
    if column == "log_total_charges":
        recommended_action = (
            "This transformed cumulative-charge feature is already more stable than the raw TotalCharges scale, so keep it and avoid ad hoc capping unless later diagnostics show a clear modeling problem."
        )
    elif outlier_count == 0:
        recommended_action = (
            "No material outlier issue is detected under the IQR rule; keep the feature as-is and handle scaling only inside the modeling pipeline if needed."
        )
    else:
        recommended_action = (
            "Review the distribution during modeling, but avoid automatic capping unless later diagnostics suggest these values behave like data errors rather than valid customer behavior."
        )
    outlier_rows.append(
        {
            "feature": column,
            "iqr_outlier_count": outlier_count,
            "iqr_outlier_share_pct": round(outlier_count / len(df) * 100, 2),
            "recommended_action": recommended_action,
        }
    )

outlier_summary_df = pd.DataFrame(outlier_rows).sort_values("iqr_outlier_share_pct", ascending=False)

scaling_guidance_df = pd.DataFrame(
    {
        "model_family": [
            "Logistic Regression",
            "Linear SVM / distance-based methods",
            "Tree ensembles (Random Forest, XGBoost, LightGBM)",
        ],
        "scaling_needed": ["Yes", "Yes", "Usually no"],
        "implementation_note": [
            "Apply scaling inside a leakage-safe sklearn pipeline.",
            "Scale numeric features inside the training pipeline only.",
            "Keep preprocessing simpler, but still encode categories consistently.",
        ],
    }
)

print("Outlier summary")
display(outlier_summary_df)
print("Scaling guidance")
display(scaling_guidance_df)


Outlier summary


,feature,iqr_outlier_count,iqr_outlier_share_pct,recommended_action
4,log_total_charges,11,0.16,This transformed cumulative-charge feature is ...
0,tenure_x_MonthlyCharges,0,0.00,No material outlier issue is detected under th...
1,MonthlyCharges,0,0.00,No material outlier issue is detected under th...
2,avg_monthly_spend,0,0.00,No material outlier issue is detected under th...
3,service_count,0,0.00,No material outlier issue is detected under th...
5,tenure,0,0.00,No material outlier issue is detected under th...


Scaling guidance


,model_family,scaling_needed,implementation_note
0,Logistic Regression,Yes,Apply scaling inside a leakage-safe sklearn pi...
1,Linear SVM / distance-based methods,Yes,Scale numeric features inside the training pip...
2,"Tree ensembles (Random Forest, XGBoost, LightGBM)",Usually no,"Keep preprocessing simpler, but still encode c..."


## Feature Selection Insights

This section bridges notebook 10 into the modeling phase by summarizing the strongest selected features, redundant families that still need care, and the handling rules that should carry into model training.

Contract-related features still need explicit redundancy control before modeling. `Contract_ordinal` and `is_month_to_month` both describe customer commitment, and notebook 10 intentionally kept one as a core feature and the other as a review feature. For linear and interpretation-focused models such as Logistic Regression, `Contract_ordinal` should usually remain the main contract signal, while `is_month_to_month` should only stay if it improves performance without destabilizing coefficients. For tree-based models, both can be tested, but they should still be treated as an overlapping family rather than as fully independent information sources.


In [39]:
feature_selection_snapshot = {
    "selected_feature_count": len(selected_feature_columns),
    "kept_feature_count": None,
    "review_feature_count": None,
    "dropped_feature_count": None,
    "top_selected_features": selected_feature_columns[:15],
}

if final_selection_summary_path.exists():
    final_selection_summary_df = pd.read_csv(final_selection_summary_path)
    if not final_selection_summary_df.empty:
        summary_row = final_selection_summary_df.iloc[0].to_dict()
        feature_selection_snapshot["kept_feature_count"] = summary_row.get("kept_feature_count")
        feature_selection_snapshot["review_feature_count"] = summary_row.get("review_feature_count")
        feature_selection_snapshot["dropped_feature_count"] = summary_row.get("dropped_feature_count")

feature_selection_summary_df = pd.DataFrame(
    {
        "metric": list(feature_selection_snapshot.keys()),
        "value": [
            ", ".join(value[:15]) if isinstance(value, list) else value
            for value in feature_selection_snapshot.values()
        ],
    }
)

likely_high_value_families_df = pd.DataFrame(
    {
        "feature_family": [
            "contract commitment",
            "customer lifecycle",
            "spend and billing",
            "support and security services",
            "payment behavior",
            "engineered interaction flags",
        ],
        "representative_features": [
            "Contract_ordinal, is_month_to_month",
            "tenure, tenure_band_new, is_new_customer",
            "MonthlyCharges, log_total_charges, avg_monthly_spend, tenure_x_MonthlyCharges",
            "OnlineSecurity_No, TechSupport_No, has_support_services, profile features",
            "payment_method_group_*, is_auto_payment",
            "contract_payment_profile_*, internet_*_profile",
        ],
        "modeling_note": [
            "Usually one of the strongest churn families; watch for redundant encodings.",
            "Important for early-churn detection and retention timing.",
            "Strong but partially redundant with tenure-driven lifecycle signal.",
            "Often highly interpretable and useful for targeting retention offers.",
            "Can reflect friction and customer behavior patterns.",
            "Retain selectively; some interaction fields may be too sparse for simple baselines.",
        ],
    }
)

print("Feature-selection snapshot from notebook 10")
display(feature_selection_summary_df)
print("Feature families to carry into modeling")
display(likely_high_value_families_df)


Feature-selection snapshot from notebook 10


,metric,value
0,selected_feature_count,53
1,kept_feature_count,32
2,review_feature_count,27
3,dropped_feature_count,18
4,top_selected_features,contract_payment_profile_One year__Bank transf...


Feature families to carry into modeling


,feature_family,representative_features,modeling_note
0,contract commitment,"Contract_ordinal, is_month_to_month",Usually one of the strongest churn families; w...
1,customer lifecycle,"tenure, tenure_band_new, is_new_customer",Important for early-churn detection and retent...
2,spend and billing,"MonthlyCharges, log_total_charges, avg_monthly...",Strong but partially redundant with tenure-dri...
3,support and security services,"OnlineSecurity_No, TechSupport_No, has_support...",Often highly interpretable and useful for targ...
4,payment behavior,"payment_method_group_*, is_auto_payment",Can reflect friction and customer behavior pat...
5,engineered interaction flags,"contract_payment_profile_*, internet_*_profile",Retain selectively; some interaction fields ma...


The current feature snapshot is cleaner than the earlier engineered table because notebook 10 already narrowed the dataset into a selected modeling set. The main decision area is now not broad schema cleanup, but disciplined handling of the remaining overlap among contract, lifecycle, and spend-related feature families.


## Business Insights

This section should read like an executive-ready summary. The purpose is to convert technical findings into practical churn meaning and retention implications.


In [40]:
manager_view_df = pd.DataFrame(
    columns=["Contract", "tenure_band", "has_support_services", "churn_rate", "segment_count", "churn_rate_pct"]
)
if engineered_df is not None and {"Contract", "tenure_band", "has_support_services", target_column}.issubset(engineered_df.columns):
    manager_view_df = (
        engineered_df.groupby(["Contract", "tenure_band", "has_support_services"], dropna=False)[target_column]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "churn_rate", "count": "segment_count"})
    )
    manager_view_df = manager_view_df.loc[manager_view_df["segment_count"] >= 40].copy()
    manager_view_df["churn_rate_pct"] = (manager_view_df["churn_rate"] * 100).round(2)
    manager_view_df = manager_view_df.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

business_takeaways_df = pd.DataFrame(columns=["business_theme", "implication"])
if not manager_view_df.empty and not combined_risk_profiles.empty:
    contract_payment_high_risk = combined_risk_profiles.iloc[0]
    contract_payment_with_support = combined_risk_profiles.iloc[1]
    contract_payment_low_risk = combined_risk_profiles.loc[
        (combined_risk_profiles["Contract"] == "Two year")
        & (combined_risk_profiles["payment_method_group"] == "auto_payment")
        & (combined_risk_profiles["has_support_services"] == 0)
    ].iloc[0]
    new_customer_high_risk = manager_view_df.loc[
        (manager_view_df["Contract"] == "Month-to-month")
        & (manager_view_df["tenure_band"] == "new")
        & (manager_view_df["has_support_services"] == 0)
    ].iloc[0]

    business_takeaways_df = pd.DataFrame(
        {
            "business_theme": [
                "Contract structure",
                "Early lifecycle churn",
                "Value-added support services",
                "Payment behavior",
                "Retention targeting",
            ],
            "implication": [
                f"Month-to-month customers paying by electronic check without support services churn at {contract_payment_high_risk['churn_rate_pct']:.2f}% across {int(contract_payment_high_risk['segment_count'])} customers, confirming that short contract commitment is the clearest retention risk.",
                f"New month-to-month customers without support services churn at {new_customer_high_risk['churn_rate_pct']:.2f}% across {int(new_customer_high_risk['segment_count'])} customers, showing that the earliest lifecycle stage needs proactive retention outreach.",
                f"Month-to-month customers paying by electronic check still churn at {contract_payment_with_support['churn_rate_pct']:.2f}% across {int(contract_payment_with_support['segment_count'])} customers even when support services are present, so support helps but does not fully offset high-risk contract and payment behavior.",
                f"Two-year customers on auto-payment without support services churn at only {contract_payment_low_risk['churn_rate_pct']:.2f}% across {int(contract_payment_low_risk['segment_count'])} customers, showing that payment stability and long commitment strongly align with retention.",
                f"The most actionable near-term retention pools are the {int(contract_payment_high_risk['segment_count'])} month-to-month electronic-check customers without support services and the {int(new_customer_high_risk['segment_count'])} new month-to-month customers without support services, both of which exceed 50% churn risk.",
            ],
        }
    )

print("High-risk business segments")
display(manager_view_df.head(15))
print("Business interpretation summary")
display(business_takeaways_df)


High-risk business segments


,Contract,tenure_band,has_support_services,churn_rate,segment_count,churn_rate_pct
6,Month-to-month,new,0,0.538462,1365,53.85
7,Month-to-month,new,1,0.459459,629,45.95
0,Month-to-month,early,0,0.380000,350,38.00
1,Month-to-month,early,1,0.374677,387,37.47
2,Month-to-month,long_term,0,0.359375,64,35.94
4,Month-to-month,mid,0,0.344482,299,34.45
5,Month-to-month,mid,1,0.320080,503,32.01
3,Month-to-month,long_term,1,0.237410,278,23.74
11,One year,long_term,1,0.137525,509,13.75
9,One year,early,1,0.127907,86,12.79


Business interpretation summary


,business_theme,implication
0,Contract structure,Month-to-month customers paying by electronic ...
1,Early lifecycle churn,New month-to-month customers without support s...
2,Value-added support services,Month-to-month customers paying by electronic ...
3,Payment behavior,Two-year customers on auto-payment without sup...
4,Retention targeting,The most actionable near-term retention pools ...


The business picture remains clear even after the selection step: the highest-risk segments are still concentrated in month-to-month customers, especially newer customers and electronic-check customers without support services, while the lowest-risk segments are anchored by two-year contracts and auto-payment behavior. Notebook 10 narrowed the modeling inputs, but it did not change the underlying business story the model should preserve.


## Modeling Recommendations

This is the bridge to the next notebook. It should clearly define how the project should move from insights into model training, validation, and comparison.


In [41]:
modeling_recommendations_df = pd.DataFrame(
    {
        "area": [
            "baseline models",
            "advanced comparisons",
            "validation strategy",
            "class imbalance handling",
            "feature inputs",
            "scaling",
            "evaluation metrics",
            "feature rules",
        ],
        "recommendation": [
            "Start with Logistic Regression as the first benchmark on the selected dataset.",
            "Compare with Random Forest, XGBoost, or LightGBM after the baseline is stable.",
            "Use stratified train/test split and stratified cross-validation.",
            "Track class imbalance explicitly; consider class_weight, threshold tuning, and PR-aware evaluation.",
            "Use the selected feature set from notebook 10 and avoid reintroducing dropped alternatives.",
            "Scale only inside the modeling pipeline for linear or distance-based models.",
            "Monitor ROC-AUC, PR-AUC, F1, recall, precision, and the confusion matrix.",
            "Guard against redundant spend/lifecycle families and overlapping contract representations.",
        ],
    }
)

modeling_recommendations_df


,area,recommendation
0,baseline models,Start with Logistic Regression as the first be...
1,advanced comparisons,"Compare with Random Forest, XGBoost, or LightG..."
2,validation strategy,Use stratified train/test split and stratified...
3,class imbalance handling,Track class imbalance explicitly; consider cla...
4,feature inputs,Use the selected feature set from notebook 10 ...
5,scaling,Scale only inside the modeling pipeline for li...
6,evaluation metrics,"Monitor ROC-AUC, PR-AUC, F1, recall, precision..."
7,feature rules,Guard against redundant spend/lifecycle famili...


## Final EDA Conclusion

The project is now in a stronger position for modeling because notebook 10 already converted the broader engineered space into a selected modeling dataset. Cleaning decisions have been documented, the main churn drivers remain visible, and the remaining modeling challenge is to compare algorithms while respecting the overlap among contract, lifecycle, and spend-related features.

The next phase should move into leakage-safe modeling with the selected feature set, a baseline classifier, a small group of stronger comparison models, and evaluation that emphasizes churn-focused metrics rather than raw accuracy alone.
